# NHL scraper

Downloads NHL play-by-play, shift charts and shooter handedness from the NHL API and writes three CSVs per season to Google Drive:

| File | One row per | Used for |
|---|---|---|
| `shots_2024.csv` | shot attempt | the xG model (`nhl_xg_modeling.ipynb`), plus the skaters on the ice for and against each shot |
| `events_2024.csv` | faceoff, penalty, hit, giveaway, takeaway | penalties drawn and taken, zone starts, and anything else you add to player stats |
| `toi_2024.csv` | player per game | ice time in seconds, split into 5v5, other even strength, power play, penalty kill |
| `stints_2024.csv` | stretch of play with the same skaters | who shared the ice and for how long: regression ratings, teammates, competition |
| `players.csv` (one file, all seasons) | player | name, position, handedness, nationality, birth date, height, weight |

**Everything is in this notebook. There is no `.py` file to upload.** Run the cells top to bottom.

- **If the shots, events and toi files already exist** and you only need the stints and player files, run sections 1 and 2, then skip to sections 7 and 8.
- Downloads are cached on Drive in `nhl_cache/`, so nothing is downloaded twice. If the runtime disconnects, set `SEASONS` in section 1 to the seasons still missing and run all again.
- With everything already cached, a full run re-reads the cache and takes about 1–2 hours (Drive is slow at reading many files). From nothing it takes 4–5 hours.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import glob, json, math, time
from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests

DATA_DIR  = '/content/drive/MyDrive'            # where the CSV files are written
CACHE_DIR = Path(f'{DATA_DIR}/nhl_cache')       # downloaded files are kept here
SEASONS   = range(2019, 2026)                   # 2019 = the 2019-20 season ... 2025 = 2025-26

BASE  = "https://api-web.nhle.com/v1"
STATS = "https://api.nhle.com/stats/rest/en"
SHOT_EVENTS = {"goal", "shot-on-goal", "missed-shot", "blocked-shot"}
GOAL_X, GOAL_Y = 89.0, 0.0                      # NHL rink coordinates: goal lines at x = +/-89 ft
SHIFT_COLS = ["shooter_toi", "own_avg_toi", "opp_min_toi", "opp_avg_toi", "opp_max_toi"]
# Non-shot events kept in the events table: (main player field, second player field)
EVENT_PLAYERS = {"faceoff": ("winningPlayerId", "losingPlayerId"),
                 "penalty": ("committedByPlayerId", "drawnByPlayerId"),
                 "hit": ("hittingPlayerId", "hitteePlayerId"),
                 "giveaway": ("playerId", None),
                 "takeaway": ("playerId", None)}
TOI_STATES = ["5v5", "ev_other", "pp", "pk"]

session = requests.Session()
session.headers.update({"User-Agent": "xg-research/0.1"})

## 2. Scraper functions

Three cells of function definitions. They only define things; nothing is downloaded until section 3.

In [ ]:
# ---- downloading (with caching and retries)
def get_json(url: str, cache_name: str | None = None, retries: int = 3, pause: float = 0.5):
    """GET a URL as JSON, with on-disk caching and simple retry/backoff."""
    if cache_name:
        path = CACHE_DIR / cache_name
        if path.exists():
            return json.loads(path.read_text())
    for attempt in range(retries):
        r = session.get(url, timeout=30)
        if r.status_code == 404:
            return None
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(2 ** attempt * 2)
            continue
        r.raise_for_status()
        data = r.json()
        if cache_name:
            CACHE_DIR.mkdir(exist_ok=True)
            (CACHE_DIR / cache_name).write_text(json.dumps(data))
        time.sleep(pause)  # be polite
        return data
    raise RuntimeError(f"Failed after {retries} attempts: {url}")


def fetch_pbp(game_id: int) -> dict | None:
    return get_json(f"{BASE}/gamecenter/{game_id}/play-by-play", f"pbp_{game_id}.json")


def fetch_shifts(game_id: int) -> list | None:
    """
    Shift chart for one game as a slim list [[playerId, teamId, period, start_sec, end_sec], ...].
    Only the slim list is cached (the raw response is ~10x larger). Empty results are not cached,
    so a game whose shifts were missing is retried on the next run.
    """
    path = CACHE_DIR / f"shifts_{game_id}.json"
    if path.exists():
        return json.loads(path.read_text())
    try:
        data = get_json(f"{STATS}/shiftcharts?cayenneExp=gameId={game_id}")
    except Exception:
        return None
    slim = []
    for sh in (data or {}).get("data", []):
        try:
            st, en = clock_to_seconds(sh["startTime"]), clock_to_seconds(sh["endTime"])
            if en > st:  # real shifts only; goal marker rows have start == end
                slim.append([int(sh["playerId"]), int(sh["teamId"]), int(sh["period"]), st, en])
        except (KeyError, TypeError, ValueError, AttributeError):
            continue
    if slim:
        CACHE_DIR.mkdir(exist_ok=True)
        path.write_text(json.dumps(slim))
    return slim or None


def fetch_handedness(player_ids) -> dict:
    """{player_id (str): 'L' / 'R'} for skaters' shooting hand. Cached in players.json."""
    path = CACHE_DIR / "players.json"
    known = json.loads(path.read_text()) if path.exists() else {}
    todo = sorted({int(p) for p in player_ids if pd.notna(p)} - {int(k) for k in known})
    for i, pid in enumerate(todo, 1):
        try:
            d = get_json(f"{BASE}/player/{pid}/landing")
            known[str(pid)] = (d or {}).get("shootsCatches")
        except Exception:
            pass  # left out of the cache, so it is retried next run
        if i % 50 == 0 or i == len(todo):
            CACHE_DIR.mkdir(exist_ok=True)
            path.write_text(json.dumps(known))
            print(f"  handedness: {i}/{len(todo)} players looked up")
    return known


def add_handedness(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    known = fetch_handedness(df["shooter_id"].unique())
    df["shooter_hand"] = df["shooter_id"].map(lambda v: known.get(str(int(v))) if pd.notna(v) else None)
    return df


def fetch_bios(player_ids) -> pd.DataFrame:
    """One row per player from the player landing endpoint. Cached in player_bios.json."""
    path = CACHE_DIR / "player_bios.json"
    known = json.loads(path.read_text()) if path.exists() else {}
    todo = sorted({int(p) for p in player_ids if pd.notna(p)} - {int(k) for k in known})
    for i, pid in enumerate(todo, 1):
        try:
            d = get_json(f"{BASE}/player/{pid}/landing") or {}
            known[str(pid)] = {
                "first_name": (d.get("firstName") or {}).get("default"),
                "last_name": (d.get("lastName") or {}).get("default"),
                "position": d.get("position"),
                "shoots_catches": d.get("shootsCatches"),
                "birth_country": d.get("birthCountry"),
                "birth_date": d.get("birthDate"),
                "height_in": d.get("heightInInches"),
                "weight_lb": d.get("weightInPounds"),
                "current_team": d.get("currentTeamAbbrev"),
                "sweater_number": d.get("sweaterNumber"),
                "is_active": d.get("isActive"),
            }
        except Exception:
            pass  # left out of the cache, so it is retried next run
        if i % 50 == 0 or i == len(todo):
            CACHE_DIR.mkdir(exist_ok=True)
            path.write_text(json.dumps(known))
            print(f"  bios: {i}/{len(todo)} players looked up")
    rows = [{"player_id": int(k), **v} for k, v in known.items()]
    df = pd.DataFrame(rows)
    if not df.empty:
        df.insert(1, "player", (df.first_name.fillna("") + " " + df.last_name.fillna("")).str.strip())
    return df


def game_ids_between(start: str, end: str, game_types=(2, 3)) -> list[int]:
    """Collect completed game IDs between two dates using the weekly schedule endpoint."""
    ids, d = set(), datetime.strptime(start, "%Y-%m-%d").date()
    end_d = datetime.strptime(end, "%Y-%m-%d").date()
    while d <= end_d:
        sched = get_json(f"{BASE}/schedule/{d.isoformat()}") or {}
        for day in sched.get("gameWeek", []):
            day_d = datetime.strptime(day["date"], "%Y-%m-%d").date()
            if start <= day["date"] <= end:
                for g in day.get("games", []):
                    if g.get("gameType") in game_types and g.get("gameState") in ("OFF", "FINAL"):
                        ids.add(g["id"])
        d += timedelta(days=7)
    return sorted(ids)


def season_game_ids(season_start_year: int, game_type: int = 2, max_games: int = 1400) -> list[int]:
    """Regular-season IDs are sequential: SSSS02 0001 .. ~1312."""
    return [int(f"{season_start_year}{game_type:02d}{n:04d}") for n in range(1, max_games + 1)]

In [ ]:
# ---- turning one game into shot rows
def clock_to_seconds(mmss: str) -> int:
    m, s = mmss.split(":")
    return int(m) * 60 + int(s)


def parse_situation(code: str | None, shooter_is_home: bool) -> dict:
    """situationCode is 4 digits: away goalie, away skaters, home skaters, home goalie."""
    if not code or len(code) != 4:
        return {"strength": None, "empty_net_against": None, "own_net_empty": None}
    away_g, away_sk, home_sk, home_g = (int(c) for c in code)
    own_sk, opp_sk = (home_sk, away_sk) if shooter_is_home else (away_sk, home_sk)
    opp_goalie_in = away_g if shooter_is_home else home_g
    own_goalie_in = home_g if shooter_is_home else away_g
    return {"strength": f"{own_sk}v{opp_sk}", "empty_net_against": int(opp_goalie_in == 0),
            "own_net_empty": int(own_goalie_in == 0)}


def attack_flip(x, shooter_is_home, home_defending_side, zone_code):
    """
    True if raw coordinates must be mirrored so the shooter attacks the net at x = +89.
    Prefer homeTeamDefendingSide; fall back to zoneCode when it's missing.
    """
    if x is None:
        return None
    flip = None
    if home_defending_side in ("left", "right"):
        # Home defends left -> home attacks right (+x). Away is the opposite.
        attacks_right = (home_defending_side == "left") == shooter_is_home
        flip = not attacks_right
    elif zone_code in ("O", "D"):
        # Offensive-zone shot with negative x means the team attacks left.
        in_left_half = x < 0
        flip = in_left_half if zone_code == "O" else not in_left_half
    return bool(flip)


def index_shifts(shifts, goalies) -> dict:
    """{period: (player_ids, team_ids, starts, ends)} as arrays, skaters only."""
    by_period = {}
    for pid, tid, per, st, en in shifts or []:
        if pid not in goalies:
            by_period.setdefault(per, []).append((pid, tid, st, en))
    return {per: tuple(np.array(c) for c in zip(*rows)) for per, rows in by_period.items()}


def shift_features(idx, period, t, shooter_id, team_id) -> dict:
    """Seconds into their current shift for the shooter, his teammates and the defenders."""
    out = dict.fromkeys(SHIFT_COLS)
    if not idx or period not in idx or t is None:
        return out
    pid, tid, st, en = idx[period]
    on = (st < t) & (en >= t)          # a player whose shift starts at t is not on yet
    toi = t - st
    me = on & (pid == shooter_id)
    if not me.any():
        me = (st <= t) & (en >= t) & (pid == shooter_id)
    if me.any():
        out["shooter_toi"] = int(toi[me].min())
    own, opp = on & (tid == team_id), on & (tid != team_id)
    if own.any():
        out["own_avg_toi"] = round(float(toi[own].mean()), 1)
    if opp.any():
        out["opp_min_toi"] = int(toi[opp].min())
        out["opp_avg_toi"] = round(float(toi[opp].mean()), 1)
        out["opp_max_toi"] = int(toi[opp].max())
    return out


def on_ice(idx, period, t, team_id, at_faceoff: bool = False):
    """
    Skater IDs on the ice for `team_id` and against it, as two "id|id|..." strings.
    A faceoff at time t is taken by the players starting at t; any other event at time t
    belongs to the players who were on up to t.
    """
    if not idx or period not in idx or t is None:
        return None, None
    pid, tid, st, en = idx[period]
    on = ((st <= t) & (en > t)) if at_faceoff else ((st < t) & (en >= t))
    join = lambda ids: "|".join(str(i) for i in sorted(ids.tolist()))
    return join(pid[on & (tid == team_id)]), join(pid[on & (tid != team_id)])


def game_toi(pbp: dict, shifts: list | None) -> pd.DataFrame:
    """
    Ice time per player for one game, in seconds, split by strength state from the player's
    side: 5v5, ev_other (4v4, 3v3), pp, pk, and other (either net empty, or anything unusual).
    The state of each second comes from counting the skaters and goalies on the ice.
    """
    if not shifts:
        return pd.DataFrame()
    home_id, away_id = pbp["homeTeam"]["id"], pbp["awayTeam"]["id"]
    abbr = {home_id: pbp["homeTeam"]["abbrev"], away_id: pbp["awayTeam"]["abbrev"]}
    roster = {p["playerId"]: p for p in pbp.get("rosterSpots", [])}
    is_goalie = lambda pid: roster.get(pid, {}).get("positionCode") == "G"

    totals = {}   # player -> [team, all, 5v5, ev_other, pp, pk]
    for per in sorted({sh[2] for sh in shifts}):
        cur = [sh for sh in shifts if sh[2] == per and sh[1] in (home_id, away_id)]
        n = max(sh[4] for sh in cur)
        sk = {home_id: np.zeros(n, int), away_id: np.zeros(n, int)}
        gl = {home_id: np.zeros(n, int), away_id: np.zeros(n, int)}
        for pid, tid, _, st, en in cur:
            (gl if is_goalie(pid) else sk)[tid][st:en] += 1
        both_goalies = (gl[home_id] > 0) & (gl[away_id] > 0)
        cum = {}
        for tid, opp in ((home_id, away_id), (away_id, home_id)):
            own, other = sk[tid], sk[opp]
            normal = both_goalies & (own >= 3) & (own <= 5) & (other >= 3) & (other <= 5)
            masks = [normal & (own == 5) & (other == 5), normal & (own == other) & (own < 5),
                     normal & (own > other), normal & (own < other)]
            cum[tid] = [np.concatenate([[0], np.cumsum(m)]) for m in masks]
        for pid, tid, _, st, en in cur:
            row = totals.setdefault(pid, [tid, 0, 0, 0, 0, 0])
            row[1] += en - st
            for k in range(4):
                row[2 + k] += int(cum[tid][k][en] - cum[tid][k][st])

    rows = []
    for pid, (tid, total, *by_state) in totals.items():
        p = roster.get(pid, {})
        name = f'{p["firstName"]["default"]} {p["lastName"]["default"]}' if p else None
        rows.append({"game_id": pbp["id"], "game_date": pbp.get("gameDate"), "player_id": pid,
                     "player": name, "pos": p.get("positionCode"), "team": abbr[tid],
                     "is_home": int(tid == home_id),
                     "opponent": abbr[away_id if tid == home_id else home_id],
                     "toi_all": total, **{f"toi_{k}": v for k, v in zip(TOI_STATES, by_state)},
                     "toi_other": total - sum(by_state)})
    return pd.DataFrame(rows)


def game_stints(game_id, shifts, info, goals, faceoff_zone) -> list:
    """
    Stints for one game: stretches of play with the same players on the ice. A new stint
    starts at every line change, goal and faceoff.

      info          {player_id: (is_home, is_goalie)}
      goals         [(game_seconds, scored_by_home), ...]
      faceoff_zone  {game_seconds: "O" / "N" / "D" from the home team's side}

    Times are seconds; a stint covers start < t <= end, so a shot or goal at time t belongs to
    the stint that ends at or after t, and a faceoff at time t starts a stint.
    """
    rows, regular = [], str(game_id)[4:6] == "02"
    for per in sorted({sh[2] for sh in shifts}):
        if regular and per > 4:      # no shootout
            continue
        cur = [sh for sh in shifts if sh[2] == per and sh[0] in info]
        if not cur:
            continue
        n, off = max(sh[4] for sh in cur), (per - 1) * 1200
        ids = np.array(sorted({sh[0] for sh in cur}))
        row_of = {p: i for i, p in enumerate(ids.tolist())}
        on_ice_grid = np.zeros((len(ids), n), bool)
        for pid, _, _, st, en in cur:
            on_ice_grid[row_of[pid], st:en] = True
        cuts = set((np.flatnonzero((on_ice_grid[:, 1:] != on_ice_grid[:, :-1]).any(axis=0)) + 1).tolist())
        cuts |= {gs - off for gs, _ in goals if 0 < gs - off < n}
        cuts |= {gs - off for gs in faceoff_zone if 0 < gs - off < n}
        bounds = [0] + sorted(cuts) + [n]
        home = np.array([info[p][0] for p in ids.tolist()])
        goalie = np.array([info[p][1] for p in ids.tolist()])
        for a, b in zip(bounds[:-1], bounds[1:]):
            on = on_ice_grid[:, a]
            hs, aw = ids[on & home & ~goalie], ids[on & ~home & ~goalie]
            if len(hs) == 0 and len(aw) == 0:
                continue
            hg, ag = bool((on & home & goalie).any()), bool((on & ~home & goalie).any())
            nh, na = len(hs), len(aw)
            if hg and ag and 3 <= nh <= 5 and 3 <= na <= 5:
                state = "5v5" if nh == na == 5 else "ev_other" if nh == na else "home_pp" if nh > na else "away_pp"
            else:
                state = "other"
            rows.append({
                "game_id": game_id, "period": per, "start_sec": a, "end_sec": b, "seconds": b - a,
                "start_gs": off + a, "end_gs": off + b, "state": state,
                "n_home": nh, "n_away": na, "home_goalie_in": int(hg), "away_goalie_in": int(ag),
                "home_score": sum(1 for gs, h in goals if h and gs <= off + a),
                "away_score": sum(1 for gs, h in goals if not h and gs <= off + a),
                "zone_start_home": faceoff_zone.get(off + a, "fly"),   # "fly" = change on the fly
                "home_skaters": "|".join(map(str, hs.tolist())),
                "away_skaters": "|".join(map(str, aw.tolist())),
            })
    return rows


def season_stints(toi: pd.DataFrame, shots: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    """Stints for every game in one season's toi table, read from the cached shift charts."""
    g = shots[shots.is_goal == 1]
    goals = {gid: list(zip(d.game_seconds.astype(int), d.is_home.astype(bool)))
             for gid, d in g.groupby("game_id")}
    fo = events[(events.event == "faceoff") & events.game_seconds.notna()].copy()
    flip = {"O": "D", "D": "O"}
    fo["zone_home"] = [z if h else flip.get(z, z) for z, h in zip(fo.zone, fo.is_home)]
    faceoffs = {gid: dict(zip(d.game_seconds.astype(int), d.zone_home)) for gid, d in fo.groupby("game_id")}

    rows = []
    for i, (gid, d) in enumerate(toi.groupby("game_id"), 1):
        path = CACHE_DIR / f"shifts_{gid}.json"
        if not path.exists():
            continue
        info = {int(p): (bool(h), pos == "G") for p, h, pos in zip(d.player_id, d.is_home, d.pos)}
        rows += game_stints(gid, json.loads(path.read_text()), info, goals.get(gid, []), faceoffs.get(gid, {}))
        if i % 200 == 0:
            print(f"  {i} games")
    return pd.DataFrame(rows)


def zone_time(hist, period, t, sign):
    """
    Seconds since play was last known to be outside the shooter's attacking zone: the time since
    the most recent faceoff or the most recent event located outside the zone, whichever is later.
    """
    if t is None:
        return None
    for p, ht, hx, hev in reversed(hist):
        if p != period:
            break
        if hev == "faceoff" or (hx is not None and hx * sign < 25):
            return t - ht
    return None


def parse_game(pbp: dict, shifts: list | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Returns (shots, events) for one game."""
    game_id = pbp["id"]
    game_date = pbp.get("gameDate")
    home_id = pbp["homeTeam"]["id"]
    away_id = pbp["awayTeam"]["id"]
    team_abbr = {home_id: pbp["homeTeam"]["abbrev"], away_id: pbp["awayTeam"]["abbrev"]}
    players = {
        p["playerId"]: {
            "name": f'{p["firstName"]["default"]} {p["lastName"]["default"]}',
            "pos": p.get("positionCode"),
            "team": p.get("teamId"),
        }
        for p in pbp.get("rosterSpots", [])
    }

    goalies = {pid for pid, p in players.items() if p["pos"] == "G"}
    shift_idx = index_shifts(shifts, goalies)

    rows, ev_rows, prev, prev2 = [], [], None, None
    name_of = lambda pid: players.get(pid, {}).get("name")
    hist = []          # (period, seconds, raw x, event) for located events and faceoffs
    recent_shots = []  # (seconds, team) of shot attempts since the last faceoff
    home_goals = away_goals = 0  # running score; the API only puts scores on goal events
    last_faceoff = None          # (period, seconds) of the latest faceoff
    cur_code, code_since = None, None   # current situationCode and when it began
    for play in pbp.get("plays", []):
        ev = play.get("typeDescKey")
        det = play.get("details", {}) or {}
        period = play.get("periodDescriptor", {}).get("number")
        period_type = play.get("periodDescriptor", {}).get("periodType")
        t = clock_to_seconds(play["timeInPeriod"]) if play.get("timeInPeriod") else None
        game_seconds = (period - 1) * 1200 + t if period and t is not None else None

        if ev == "faceoff":
            last_faceoff = (period, t)
            recent_shots = []
        if play.get("situationCode") != cur_code:
            cur_code, code_since = play.get("situationCode"), (period, t)

        # Team that took the action. For a blocked shot the API's event owner may be the
        # blocking team, so the shooter's team is read from the roster instead.
        owner = det.get("eventOwnerTeamId")
        act_team = owner
        if ev == "blocked-shot":
            act_team = players.get(det.get("shootingPlayerId"), {}).get("team")
            if act_team not in (home_id, away_id):   # roster lookup failed: assume owner = blocker
                act_team = away_id if owner == home_id else home_id

        if ev in SHOT_EVENTS and period_type != "SO":
            team_id = act_team
            is_home = team_id == home_id
            shooter_id = det.get("scoringPlayerId") or det.get("shootingPlayerId")
            x_raw, y_raw = det.get("xCoord"), det.get("yCoord")
            zone = det.get("zoneCode")
            # zoneCode is from the event owner's view; put it in the shooter's view.
            if owner != team_id and zone in ("O", "D"):
                zone = "O" if zone == "D" else "D"
            flip = attack_flip(x_raw, is_home, play.get("homeTeamDefendingSide"), zone)
            sign = -1 if flip else 1
            x = x_raw * sign if x_raw is not None and y_raw is not None else None
            y = y_raw * sign if x is not None else None
            sit = parse_situation(play.get("situationCode"), is_home)
            same_period = bool(prev and prev["period"] == period)
            same_period2 = bool(prev2 and prev2["period"] == period)
            mine = [st for st, tm in recent_shots if tm == team_id and t is not None and st is not None]

            dist = angle = None
            if x is not None:
                dist = math.hypot(GOAL_X - x, y - GOAL_Y)
                angle = math.degrees(math.atan2(abs(y), GOAL_X - x))  # 0 = straight on

            row = {
                "game_id": game_id,
                "event_id": play.get("eventId"),
                "period": period,
                "period_type": period_type,
                "time_in_period": play.get("timeInPeriod"),
                "game_seconds": game_seconds,
                "event": ev,
                "is_goal": int(ev == "goal"),
                "team": team_abbr.get(team_id),
                "is_home": int(is_home),
                "home_team": team_abbr.get(home_id),
                "away_team": team_abbr.get(away_id),
                "game_date": game_date,
                "shooter_id": shooter_id,
                "shooter": players.get(shooter_id, {}).get("name"),
                "shooter_pos": players.get(shooter_id, {}).get("pos"),
                "goalie_id": det.get("goalieInNetId"),
                "assist1_id": det.get("assist1PlayerId"),
                "assist2_id": det.get("assist2PlayerId"),
                "blocker_id": det.get("blockingPlayerId"),
                "shot_type": det.get("shotType"),
                # miss reason: wide-left, wide-right, above-crossbar, hit-post, short, ...
                # "short" misses only exist from 2023-24 on; exclude them from xG training
                "miss_reason": det.get("reason") if ev == "missed-shot" else None,
                "x_raw": x_raw, "y_raw": y_raw,
                "x": x, "y": y,
                "distance": dist, "angle": angle,
                "zone": zone,  # shooter's perspective
                "situation_code": play.get("situationCode"),
                **sit,
                # score *before* this shot, so goals don't leak into their own feature
                "home_score": home_goals,
                "away_score": away_goals,
                "score_diff": (home_goals - away_goals) if is_home else (away_goals - home_goals),
                # previous-event context (useful xG features)
                "prev_event": prev["event"] if prev else None,
                "prev_short_miss": int(bool(prev and prev["event"] == "missed-shot"
                                            and prev["reason"] == "short")),
                "prev_team_same": (int(prev["team_id"] == team_id) if prev and prev["team_id"] else None),
                "sec_since_prev": (game_seconds - prev["game_seconds"]
                                   if prev and game_seconds is not None and prev["game_seconds"] is not None
                                   and prev["period"] == period else None),
                "prev_x_raw": prev["x"] if prev else None,
                "prev_y_raw": prev["y"] if prev else None,
                # previous event's location in the shooter's frame (attacking net at x = +89)
                "prev_x": (prev["x"] * sign if same_period and x is not None
                           and prev["x"] is not None else None),
                "prev_y": (prev["y"] * sign if same_period and x is not None
                           and prev["y"] is not None else None),
                "sec_since_faceoff": (t - last_faceoff[1] if last_faceoff and t is not None
                                      and last_faceoff[0] == period and last_faceoff[1] is not None else None),
                # time since the strength state last changed (e.g. since a power play began)
                "sec_since_state_change": (t - code_since[1] if code_since and t is not None
                                           and code_since[0] == period and code_since[1] is not None else None),
                **shift_features(shift_idx, period, t, shooter_id, team_id),
                # the event before the previous one
                "prev2_event": prev2["event"] if prev2 else None,
                "prev2_team_same": (int(prev2["team_id"] == team_id) if prev2 and prev2["team_id"] else None),
                "sec_since_prev2": (game_seconds - prev2["game_seconds"]
                                    if same_period2 and game_seconds is not None
                                    and prev2["game_seconds"] is not None else None),
                "prev2_x": (prev2["x"] * sign if same_period2 and x is not None
                            and prev2["x"] is not None else None),
                "prev2_y": (prev2["y"] * sign if same_period2 and x is not None
                            and prev2["y"] is not None else None),
                # sustained pressure
                "sec_in_zone": zone_time(hist, period, t, sign) if x is not None else None,
                "team_shots_10s": sum(t - st <= 10 for st in mine),
                "team_shots_30s": sum(t - st <= 30 for st in mine),
            }
            row["on_ice_for"], row["on_ice_against"] = on_ice(shift_idx, period, t, team_id)
            rows.append(row)

        elif ev in EVENT_PLAYERS and period_type != "SO":
            f1, f2 = EVENT_PLAYERS[ev]
            p1, p2 = det.get(f1), (det.get(f2) if f2 else None)
            o_home = owner == home_id
            on_for, on_against = on_ice(shift_idx, period, t, owner, at_faceoff=(ev == "faceoff"))
            ev_rows.append({
                "game_id": game_id, "game_date": game_date, "event_id": play.get("eventId"),
                "period": period, "period_type": period_type,
                "time_in_period": play.get("timeInPeriod"), "game_seconds": game_seconds,
                "event": ev,
                # team: faceoff winner / penalized team / hitter / player who gave or took the puck
                "team": team_abbr.get(owner), "is_home": int(o_home),
                "home_team": team_abbr.get(home_id), "away_team": team_abbr.get(away_id),
                "player_id": p1, "player": name_of(p1),        # winner / penalized / hitter / puck carrier
                "player2_id": p2, "player2": name_of(p2),      # loser / drew the penalty / player hit
                "x_raw": det.get("xCoord"), "y_raw": det.get("yCoord"),
                "zone": det.get("zoneCode"),                   # O / N / D from `team`'s side
                "situation_code": play.get("situationCode"),
                **parse_situation(play.get("situationCode"), o_home),
                "penalty_type": det.get("descKey") if ev == "penalty" else None,
                "penalty_class": det.get("typeCode") if ev == "penalty" else None,
                "penalty_minutes": det.get("duration") if ev == "penalty" else None,
                "served_by_id": det.get("servedByPlayerId") if ev == "penalty" else None,
                "on_ice_for": on_for, "on_ice_against": on_against,
            })

        if ev == "goal" and period_type != "SO":
            home_goals = det.get("homeScore", home_goals)
            away_goals = det.get("awayScore", away_goals)

        if ev in SHOT_EVENTS and period_type != "SO":
            recent_shots.append((t, act_team))
        if t is not None and (ev == "faceoff" or det.get("xCoord") is not None):
            hist.append((period, t, det.get("xCoord"), ev))

        prev2 = prev
        prev = {
            "event": ev,
            "team_id": act_team,
            "game_seconds": game_seconds,
            "period": period,
            "x": det.get("xCoord"),
            "y": det.get("yCoord"),
            "reason": det.get("reason"),
        }

    df = pd.DataFrame(rows)
    if not df.empty:
        # Distance travelled since the previous event (raw coords, same frame for both points)
        df["dist_from_prev"] = ((df["x_raw"] - df["prev_x_raw"]) ** 2 +
                                (df["y_raw"] - df["prev_y_raw"]) ** 2) ** 0.5
        df["rebound"] = ((df["prev_event"].isin(SHOT_EVENTS - {"goal"})) &
                         (df["prev_team_same"] == 1) &
                         (df["sec_since_prev"] <= 3)).astype(int)
    return df, pd.DataFrame(ev_rows)

In [ ]:
# ---- scraping a list of games
def scrape_games(game_ids, stop_after_misses: int = 10,
                 with_shifts: bool = True, with_hand: bool = True) -> dict:
    """Returns {"shots": ..., "events": ..., "toi": ...} for the given games."""
    shots, events, toi, misses, no_shifts = [], [], [], 0, 0
    for i, gid in enumerate(game_ids, 1):
        pbp = fetch_pbp(gid)
        if pbp is None or not pbp.get("plays"):
            misses += 1
            if misses >= stop_after_misses:
                print(f"{misses} missing games in a row; stopping at {gid}.")
                break
            continue
        misses = 0
        shifts = fetch_shifts(gid) if with_shifts else None
        no_shifts += int(with_shifts and not shifts)
        sh, ev = parse_game(pbp, shifts)
        shots.append(sh); events.append(ev); toi.append(game_toi(pbp, shifts))
        if i % 25 == 0:
            print(f"  {i} games processed")
    cat = lambda frames: (pd.concat([f for f in frames if not f.empty], ignore_index=True)
                          if any(not f.empty for f in frames) else pd.DataFrame())
    out = {"shots": cat(shots), "events": cat(events), "toi": cat(toi)}
    if with_shifts and shots:
        print(f"  shift data missing for {no_shifts} of {len(shots)} games")
    if with_hand:
        out["shots"] = add_handedness(out["shots"])
    return out

## 3. Test on one game

Check before the long run:

1. It prints `shift data missing for 0 of 1 games`.
2. **Shots:** `on_ice_for` and `on_ice_against` each list five player IDs at 5v5.
3. **Events:** faceoffs show a winner and a loser, and the penalty rows show a type and minutes.
4. **Ice time:** the busiest skaters are around 1,200–1,700 seconds (20–28 minutes), mostly at 5v5, and each team's starting goalie is near 3,600.

In [ ]:
out = scrape_games([2024020001])
s, e, toi = out['shots'], out['events'], out['toi']
print(len(s), "shot rows,", len(e), "event rows,", len(toi), "player rows")

display(s[['time_in_period', 'event', 'team', 'shooter', 'strength', 'on_ice_for', 'on_ice_against']].head(5))
display(e[['time_in_period', 'event', 'team', 'player', 'player2', 'zone', 'penalty_type',
           'penalty_minutes', 'on_ice_for']].head(8))
display(e[e.event == 'penalty'][['time_in_period', 'team', 'player', 'player2', 'penalty_type', 'penalty_minutes']])
toi.sort_values('toi_all', ascending=False)[['player', 'pos', 'team', 'toi_all', 'toi_5v5', 'toi_ev_other',
                                             'toi_pp', 'toi_pk', 'toi_other']].head(10)

## 4. Scrape every season

Writes `shots_YYYY.csv`, `events_YYYY.csv` and `toi_YYYY.csv` for each season, replacing any earlier files of the same name. Each season stops by itself after the last game (the "10 missing games in a row" message is normal).

In [ ]:
for y in SEASONS:
    print(f"--- {y}-{str(y + 1)[2:]} ---")
    out = scrape_games(season_game_ids(y))
    for name, table in out.items():
        table.to_csv(f'{DATA_DIR}/{name}_{y}.csv', index=False)
    print(f"saved {y}: {len(out['shots']):,} shots, {len(out['events']):,} events, "
          f"{len(out['toi']):,} player-games from {out['shots'].game_id.nunique()} games\n")

## 5. Check the output

One row per season.

- `on_ice` is the share of shots with on-ice players attached. It is below 1.00 only for games with no shift chart on the NHL's side; those games also have no rows in the ice-time file.
- `min_5v5_per_game` is 5v5 minutes per game worked out from the ice-time file. It should be roughly 46–50 in every season. A value far from that means the strength-state split is off.
- `penalties` and `faceoffs` are counts from the events file, as a sanity check (about 7–9 penalties and 55–65 faceoffs per game).

Any shots file reported as **OLD FORMAT** was made by an earlier scraper. Delete it from Drive, or run section 4 again for that season.

In [ ]:
rows = []
for f in sorted(glob.glob(f'{DATA_DIR}/shots_*.csv')):
    y = Path(f).stem.split('_', 1)[1]
    d = pd.read_csv(f, low_memory=False)
    if 'on_ice_for' not in d.columns or not Path(f'{DATA_DIR}/toi_{y}.csv').exists():
        print(f"OLD FORMAT: {Path(f).name}")
        continue
    ev, toi = pd.read_csv(f'{DATA_DIR}/events_{y}.csv', low_memory=False), pd.read_csv(f'{DATA_DIR}/toi_{y}.csv')
    games = d.game_id.nunique()
    sk = toi[toi.pos != 'G']
    rows.append({'season': y, 'games': games, 'shots': len(d), 'goals': int(d.is_goal.sum()),
                 'duplicates': int(d.duplicated(['game_id', 'event_id']).sum()),
                 'on_ice': round(d.on_ice_for.notna().mean(), 3),
                 'hand': round(d.shooter_hand.notna().mean(), 3),
                 'games_with_toi': toi.game_id.nunique(),
                 'min_5v5_per_game': round(sk.toi_5v5.sum() / 10 / 60 / max(toi.game_id.nunique(), 1), 1),
                 'penalties': round((ev.event == 'penalty').sum() / games, 1),
                 'faceoffs': round((ev.event == 'faceoff').sum() / games, 1)})
pd.DataFrame(rows)

## 6. Optional: the season in progress

Adds finished games from the current season as `shots_2026.csv`, `events_2026.csv` and `toi_2026.csv`. Leave `SCRAPE_CURRENT = False` unless you want them: once the shots file exists, the modeling notebook will include 2026-27 as a partial season.

Uses the schedule to pick up only finished games, so a game still being played is never cached.

In [ ]:
SCRAPE_CURRENT = False
CURRENT, SEASON_START = 2026, "2026-10-01"

if SCRAPE_CURRENT:
    ids = game_ids_between(SEASON_START, (date.today() - timedelta(days=1)).isoformat(), game_types=(2,))
    ids = [g for g in ids if str(g).startswith(str(CURRENT))]
    out = scrape_games(ids, stop_after_misses=len(ids) + 1)
    for name, table in out.items():
        table.to_csv(f'{DATA_DIR}/{name}_{CURRENT}.csv', index=False)
    print(f"saved {CURRENT}: {len(out['shots']):,} shots from {out['shots'].game_id.nunique()} games")

## 7. Stints

Builds `stints_YYYY.csv` for every season that has a `toi_YYYY.csv`. A stint is a stretch of play with the same players on the ice; a new one starts at every line change, goal and faceoff. Each row has its length in seconds, the home and away skaters, the strength state, the score when it began, and the zone it started in from the home team's side (`fly` means a change on the fly).

This reads only the small cached shift files and the CSVs from section 4, not the play-by-play, so it is much quicker than section 4. To attach shots to stints, match on `game_id` and `start_gs < game_seconds <= end_gs`.

The last column printed, 5v5 minutes per game, should match the `min_5v5_per_game` value from section 5.

In [ ]:
for f in sorted(glob.glob(f'{DATA_DIR}/toi_*.csv')):
    y = Path(f).stem.split('_', 1)[1]
    toi = pd.read_csv(f, usecols=['game_id', 'player_id', 'pos', 'is_home'])
    shots = pd.read_csv(f'{DATA_DIR}/shots_{y}.csv', usecols=['game_id', 'is_goal', 'game_seconds', 'is_home'])
    events = pd.read_csv(f'{DATA_DIR}/events_{y}.csv', usecols=['game_id', 'event', 'game_seconds', 'zone', 'is_home'])
    print(f"--- {y} ---")
    st = season_stints(toi, shots, events)
    st.to_csv(f'{DATA_DIR}/stints_{y}.csv', index=False)
    games = st.game_id.nunique()
    print(f"saved stints_{y}.csv: {len(st):,} stints from {games} games, "
          f"{st[st.state == '5v5'].seconds.sum() / 60 / games:.1f} min of 5v5 per game\n")

## 8. Player details

Looks up every player who appears in the ice-time files (skaters and goalies) and writes `players.csv`: name, position, handedness, nationality, birth date, height, weight, current team and number. Lookups are cached, so a second run only fetches new players. The first run makes one request per player, about 2,000–2,500 in all.

In [ ]:
ids = pd.concat([pd.read_csv(f, usecols=['player_id']) for f in glob.glob(f'{DATA_DIR}/toi_*.csv')]).player_id.unique()
players = fetch_bios(ids)
players.to_csv(f'{DATA_DIR}/players.csv', index=False)
print(f"saved players.csv: {len(players):,} players")
print(players.position.value_counts().to_dict())
print(players.birth_country.value_counts().head(8).to_dict())
players.head()

In [ ]:
toi = pd.read_csv(f'{DATA_DIR}/toi_2023.csv')
st = pd.read_csv(f'{DATA_DIR}/stints_2023.csv', usecols=['game_id', 'state', 'seconds'])
gap = st[st.state == '5v5'].groupby('game_id').seconds.sum() - toi[toi.pos != 'G'].groupby('game_id').toi_5v5.sum() / 10
print((gap.abs() > 1).mean().round(3), "of games differ;", gap.describe().round(1).to_dict())

In [ ]:
gid = gap.idxmax()
d = pd.DataFrame(json.loads((CACHE_DIR / f'shifts_{gid}.json').read_text()),
                 columns=['player', 'team', 'period', 'start', 'end']).sort_values(['player', 'period', 'start'])
d['overlap'] = d.start < d.groupby(['player', 'period']).end.shift()
print(gid, "-", int(d.overlap.sum()), "overlapping shift records")
d[d.overlap].head(10)